# Trading Agents Orchestrator
## Automated Weekly Financial Analysis & Options Trading Workflow

This notebook orchestrates a multi-agent system that:
1. Fetches global news from OpenClaw
2. Analyzes sentiment from Reddit
3. Calculates ETF sector impacts
4. Makes trading decisions
5. Executes trades via Interactive Brokers

**Runs every: Monday at 9:00 AM**

## Setup & Configuration

In [ ]:
# System Path Setup
import sys
import os

# Add project directories to path
project_root = "/Users/eladzoarets/Projects/agents/7_trading_agents"
sys.path.insert(0, project_root)
sys.path.insert(0, os.path.join(project_root, "config"))
sys.path.insert(0, os.path.join(project_root, "utils"))
sys.path.insert(0, os.path.join(project_root, "agents"))

os.chdir(project_root)
print(f"Working directory: {os.getcwd()}")
print(f"Project root: {project_root}")

In [ ]:
# Import Required Libraries
import asyncio
import json
from datetime import datetime, timedelta
import pandas as pd
import logging
from typing import Dict, List

# Trading agents
from agents.news_agent import NewsAggregationAgent
from agents.sentiment_agent import SentimentAnalysisAgent
from agents.etf_analyzer import ETFImpactAnalyzerAgent
from agents.decision_and_executor import TradingDecisionAgent, InteractiveBrokersExecutor

# Utilities
from utils.openclaw_client import OpenClawClient
from utils.etf_mappings import get_etf_info, get_etfs_by_sector, Sector
from config.config import TARGET_ETFS, TIMEZONE, MIN_CONFIDENCE_THRESHOLD

# Setup Logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

print("✓ All libraries imported successfully")
print(f"✓ Timezone: {TIMEZONE}")
print(f"✓ Min confidence threshold: {MIN_CONFIDENCE_THRESHOLD}")

## Step 1: News Aggregation from OpenClaw

In [ ]:
print("\n" + "="*60)
print("STEP 1: NEWS AGGREGATION FROM OPENCLAW")
print("="*60)

# Initialize News Agent
news_agent = NewsAggregationAgent()

# Fetch news focusing on key sectors
sectors_to_analyze = ["energy", "technology", "finance", "commodities"]

news_result = await news_agent.run(
    sectors=sectors_to_analyze,
    urgency="high"
)

print(f"\n✓ News Aggregation Complete")
print(f"Timestamp: {news_result['timestamp']}")
print(f"\nAnalysis Output:")
print(json.dumps(news_result['analysis'], indent=2)[:500])

## Step 2: Sentiment Analysis from Reddit

In [ ]:
print("\n" + "="*60)
print("STEP 2: SENTIMENT ANALYSIS FROM REDDIT")
print("="*60)

# Initialize Sentiment Agent
sentiment_agent = SentimentAnalysisAgent()

# Analyze sentiment for target sectors
sentiment_result = await sentiment_agent.run(
    sectors=["energy", "technology", "finance"],
    keywords=["XLE", "XLK", "XLF", "oil", "tech", "stock market"]
)

print(f"\n✓ Sentiment Analysis Complete")
print(f"Timestamp: {sentiment_result['timestamp']}")
print(f"\nSentiment Analysis Output:")
print(json.dumps(sentiment_result['sentiment_analysis'], indent=2)[:500])

## Step 3: ETF Impact Analysis

In [ ]:
print("\n" + "="*60)
print("STEP 3: ETF IMPACT ANALYSIS")
print("="*60)

# Initialize ETF Analyzer
etf_analyzer = ETFImpactAnalyzerAgent()

# Analyze impact of news on ETF sectors
impact_result = await etf_analyzer.run(
    event_classification=news_result['analysis'],
    sentiment_data=sentiment_result['sentiment_analysis'],
    etf_symbols=['XLE', 'XLK', 'XLF', 'XLY', 'XLU']
)

print(f"\n✓ ETF Impact Analysis Complete")
print(f"Timestamp: {impact_result['timestamp']}")
print(f"\nImpact Analysis Output:")
print(json.dumps(impact_result['impact_analysis'], indent=2)[:500])

## Step 4: Trading Decisions

In [ ]:
print("\n" + "="*60)
print("STEP 4: TRADING DECISIONS")
print("="*60)

# Initialize Decision Agent
decision_agent = TradingDecisionAgent()

# Mock ETF prices (in production, fetch from yfinance)
etf_prices = {
    "XLE": {"current_price": 92.50, "volatility": 0.28},
    "XLK": {"current_price": 148.75, "volatility": 0.18},
    "XLF": {"current_price": 41.20, "volatility": 0.22},
    "XLY": {"current_price": 82.10, "volatility": 0.25},
    "XLU": {"current_price": 68.75, "volatility": 0.15},
}

# Make trading decisions
trade_decisions = await decision_agent.run(
    impact_analysis=impact_result['impact_analysis'],
    sentiment_data=sentiment_result['sentiment_analysis'],
    etf_prices=etf_prices
)

print(f"\n✓ Trading Decisions Complete")
print(f"Timestamp: {trade_decisions['timestamp']}")
print(f"\nTrade Decisions:")
print(json.dumps(trade_decisions['trade_decisions'], indent=2)[:800])

## Step 5: Execute Trades via Interactive Brokers

In [ ]:
print("\n" + "="*60)
print("STEP 5: EXECUTE TRADES VIA INTERACTIVE BROKERS")
print("="*60)

# Initialize Executor
executor = InteractiveBrokersExecutor()

# Extract trade list from decisions
trades_to_execute = trade_decisions['trade_decisions'].get('trades', [])

print(f"\nTrades to execute: {len(trades_to_execute)}")

if trades_to_execute:
    # Execute trades
    execution_result = await executor.run(trades_to_execute)
    
    print(f"\n✓ Trade Execution Complete")
    print(f"Trades executed: {execution_result['trades_executed']}")
    print(f"\nExecution Results:")
    for trade in execution_result['executed_trades']:
        if trade.get('status') == 'EXECUTED':
            print(f"  ✓ Order #{trade['order_id']}: {trade['etf_symbol']} {trade['trade_type']} @ ${trade['execution_price']:.2f}")
else:
    print("\n⚠ No trades met confidence threshold for execution")
    execution_result = {"trades_executed": 0, "executed_trades": []}

## Summary & Report Generation

In [ ]:
print("\n" + "="*60)
print("WORKFLOW SUMMARY")
print("="*60)

summary = {
    "workflow_timestamp": datetime.now().isoformat(),
    "news_analyzed": news_result.get('status', 'unknown'),
    "sentiment_analyzed": sentiment_result.get('status', 'unknown'),
    "impact_calculated": impact_result.get('status', 'unknown'),
    "trades_decided": len(trade_decisions.get('trade_decisions', {}).get('trades', [])),
    "trades_executed": execution_result.get('trades_executed', 0),
    "next_review": (datetime.now() + timedelta(days=7)).strftime("%A, %Y-%m-%d at 09:00 AM")
}

print(json.dumps(summary, indent=2))

# Save summary to logs
log_file = "./reports/workflow_summary.json"
os.makedirs(os.path.dirname(log_file), exist_ok=True)
with open(log_file, 'w') as f:
    json.dump(summary, f, indent=2)
    
print(f"\n✓ Summary saved to {log_file}")

## Position Tracking

In [ ]:
# Create DataFrame of executed trades for tracking
if execution_result['executed_trades']:
    trades_df = pd.DataFrame(execution_result['executed_trades'])
    print("\nExecuted Positions:")
    print(trades_df[['order_id', 'etf_symbol', 'trade_type', 'strike_price', 'quantity', 'status']].to_string())
    
    # Save to CSV for tracking
    trades_df.to_csv("./reports/executed_trades.csv", index=False, mode='a')
    print(f"\n✓ Positions saved to ./reports/executed_trades.csv")
else:
    print("\nNo trades were executed in this run.")

## Next Steps

### Automatic Scheduling
To run this workflow automatically every Monday at 9:00 AM, use APScheduler:

```python
from apscheduler.schedulers.background import BackgroundScheduler

scheduler = BackgroundScheduler()
scheduler.add_job(lambda: asyncio.run(orchestrator_run()), 'cron', day_of_week='mon', hour=9)
scheduler.start()
```

### Weekly Position Review
- Check active positions every Monday morning
- Evaluate P&L against targets
- Decide: HOLD, CLOSE, or ADD positions
- Run new analysis for next week

### Risk Management
- Max position size: 2% of account per trade
- Stop loss: 5% per position
- Take profit: 15% per position
- Max concurrent positions: 10